# Customer Churn Forecasting - Random Forest + SMOTE
## Module 6: Ensemble Classification with Oversampling

Random Forest ensemble classifier combined with SMOTE (Synthetic Minority Over-sampling Technique) to handle class imbalance in churn prediction.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)

# Try importing SMOTE
try:
    from imblearn.over_sampling import SMOTE
    print('imbalanced-learn loaded successfully.')
except ImportError:
    print('Installing imbalanced-learn...')
    import subprocess
    subprocess.check_call(['pip', 'install', 'imbalanced-learn', '-q'])
    from imblearn.over_sampling import SMOTE
    print('imbalanced-learn installed and loaded.')

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('All libraries loaded successfully.')

## 6.1 Load and Prepare Data

In [ ]:
# Load and prepare the dataset
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Preprocessing
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature engineering
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in df.columns)
df['ChargesPerService'] = np.where(df['TotalServices'] > 0, df['MonthlyCharges'] / df['TotalServices'], df['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

# Encode
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

# Split
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train churn distribution: {dict(y_train.value_counts())}')

## 6.2 Apply SMOTE Oversampling

In [ ]:
# Apply SMOTE on training data only
print('Before SMOTE:')
print(f'  Retained: {sum(y_train == 0):,} | Churned: {sum(y_train == 1):,}')
print(f'  Churn ratio: {sum(y_train == 1) / len(y_train) * 100:.2f}%')

smote = SMOTE(sampling_strategy=0.5, random_state=42)  # 50% minority ratio
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(f'  Retained: {sum(y_train_smote == 0):,} | Churned: {sum(y_train_smote == 1):,}')
print(f'  Churn ratio: {sum(y_train_smote == 1) / len(y_train_smote) * 100:.2f}%')
print(f'  Total samples: {len(X_train_smote):,} (was {len(X_train):,})')

# Visualize the resampling effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd.Series(y_train).value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Before SMOTE', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)
axes[0].set_ylabel('Count')

pd.Series(y_train_smote).value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('After SMOTE', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Retained', 'Churned'], rotation=0)
axes[1].set_ylabel('Count')

plt.suptitle('SMOTE Resampling Effect on Training Data', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.3 Train Random Forest Classifier

In [ ]:
# Train Random Forest on SMOTE-resampled data
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_smote, y_train_smote)

# Predictions on original (unsampled) test set
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print('=' * 60)
print('RANDOM FOREST + SMOTE - RESULTS')
print('=' * 60)
print(f'\nAccuracy:  {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred_rf)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred_rf)*100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred_rf):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}')

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_test, y_pred_rf, target_names=['Retained', 'Churned']))

## 6.4 Confusion Matrix

In [ ]:
# Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            annot_kws={'size': 16}, linewidths=1, linecolor='white')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Random Forest + SMOTE - Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm_rf.ravel()
print(f'True Negatives:  {tn:,} | False Positives: {fp:,}')
print(f'False Negatives: {fn:,} | True Positives:  {tp:,}')

## 6.5 Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 15 Most Important Features (Random Forest):')
display(rf_importance.head(15))

# Visualize
top_15_rf = rf_importance.head(15)
plt.figure(figsize=(10, 7))
plt.barh(range(len(top_15_rf)), top_15_rf['Importance'].values, 
         color='#8e44ad', edgecolor='black', alpha=0.85)
plt.yticks(range(len(top_15_rf)), top_15_rf['Feature'].values)
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Top 15 Feature Importances - Random Forest + SMOTE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, color='#8e44ad', lw=2, label=f'RF + SMOTE (AUC = {auc_rf:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.fill_between(fpr_rf, tpr_rf, alpha=0.1, color='#8e44ad')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Random Forest + SMOTE', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rf_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()